In [1]:
!pip install -U langchain faiss-cpu sentence-transformers huggingface_hub dspy-ai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install accelerate peft transformers datasets -q

In [3]:
!pip install -U langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00


In [4]:
!pip install -U langchain-huggingface -q

In [ ]:
!pip install --upgrade triton -q

In [5]:
!pip install bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.8 MB/s eta 0:00:00


In [6]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00


In [7]:
!pip install rouge_score -q

  Preparing metadata (setup.py) ... done


In [8]:
!pip install bert_score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.5 MB/s eta 0:00:00


In [9]:
!pip install sentence_transformers -q

In [10]:
!pip install protobuf -q

In [11]:
!pip install sentencepiece -q

In [12]:
!pip install hf_xet

In [13]:
!pip install gradio -q

In [14]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
CUDA device count: 1
Device name: Tesla T4


## Chunking the text

In [15]:
import warnings
warnings.filterwarnings("ignore")

In [16]:
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Load biomedical passages dataset (passages with passage IDs)
df_passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

# Clean data
df_passages = df_passages.dropna(subset=["passage"]).drop_duplicates(subset=["passage"])

# Create Documents with metadata = passage ID (as string)
documents = []
for _, row in df_passages.iterrows():
    documents.append(
        Document(
            page_content=row["passage"],
            metadata={"id": str(row.name)}
        )
    )

# Chunk documents (512 tokens per chunk, 100 overlap)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
chunked_docs = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunked_docs)}")

Total chunks created: 111480


## Building and Saving FAISS Vector Store

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Use sentence-transformers lightweight embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "all-MiniLM-L6-v2"
print(f"Using device: {device}")

embedding_model = SentenceTransformer(embedding_model_name, device=device)
embedding_wrapper = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs={"device": device})

In [18]:
from tqdm.auto import tqdm

all_embeddings = []
batch_size = 64

for i in tqdm(range(0, len(chunked_docs), batch_size), desc="Embedding batches"):
    batch = chunked_docs[i:i+batch_size]
    texts = [doc.page_content for doc in batch]
    embeddings = embedding_model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    all_embeddings.extend(embeddings)


print(f"Embedded all {len(all_embeddings)} chunks")

# Build FAISS index from embeddings and docs with metadata
text_embedding_pairs = [(doc.page_content, embedding) for doc, embedding in zip(chunked_docs, all_embeddings)]

vectorstore = FAISS.from_embeddings(text_embedding_pairs, embedding_wrapper)
vectorstore.save_local("bio_faiss_index")

Embedding batches:   0%|          | 0/1742 [00:00<?, ?it/s]

Embedded all 111480 chunks


### Langchain

In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
import torch

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    torch_dtype=torch.float16
)
pipe = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

# Load FAISS index with embedding wrapper, allow deserialization
vectorstore = FAISS.load_local(
    "bio_faiss_index",
    embeddings=embedding_wrapper,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever,
    return_source_documents=True
)

In [27]:
OOD_KEYWORDS = ["economy", "inflation", "tariff", "stock market", "war", "climate"]

def is_out_of_domain(query):
    return any(word in query.lower() for word in OOD_KEYWORDS)

def safe_biomedical_qa(query):
    if is_out_of_domain(query):
        return "Sorry, I am only trained to answer biomedical-related questions."
    output = qa_chain.invoke({"query": query})
    return output['result']

In [28]:
print(safe_biomedical_qa("What is the role of TNF-alpha in inflammation?"))

TNF-alpha acts as a potent activator of Nox1-based oxidase in colon epithelial cells, suggesting a potential role of this oxidase in inflammation of the colon. ------------


In [29]:
print(safe_biomedical_qa("What are the effects of tariffs on the economy?"))

Sorry, I am only trained to answer biomedical-related questions.
